# Notebook 05 — Explore your data with Genie

**What you’ll learn:** How to interact with Genie through its chat UI and verify that it returns correct answers by comparing with direct SQL.

**How this works:** You’ll open your Genie space in the browser and ask questions in plain English. Then you’ll run reference SQL queries here to confirm Genie’s answers match the ground truth.

**Before you start:** Run notebooks **02** (data) and **03** (Genie spaces).

**Compute:** Serverless.

## Compute

Use **Serverless** (recommended).


## Configuration

Use the **same** **Catalog** and **Schema** as **01** / **02**. This notebook **builds** the Genie link from your workspace and the space id in `workshop_config` so the link stays valid.


In [ ]:
%run ./00_workshop_config

In [ ]:
import re
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
host = w.config.host.rstrip("/")

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
if not PRIMARY_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 02 first.")
print(f"Configured: {genie_ui_room_url(PRIMARY_SPACE_ID)}")


## Quality and OEE — try in Genie, then compare

1. **What is the average OEE by plant for 2024?**
   - **Expected answer:** One row per plant with **average OEE %** = `AVG(oee_score) * 100` over `quality_metrics_daily` rows where `YEAR(date) = 2024`, joined to `plants` for the name. Ordering may vary; values should match the reference table below.

2. **Which production lines have the highest defect rates in 2024?**
   - **Expected answer:** Defect rate **%** = `COUNT(defect_detected) / COUNT(unit_produced) * 100` on `production_events` in 2024, grouped by line (join `production_lines`). Highest rates appear at the **top** when sorted descending.

3. **Show first pass yield trend by month for 2024.**
   - **Expected answer:** One row per **month** with **avg FPY %** = `AVG(first_pass_yield) * 100` from `quality_metrics_daily` where `YEAR(date)=2024`, e.g. `DATE_TRUNC('month', date)`.


## Production and downtime — try in Genie, then compare

1. **Compare total downtime minutes across plants.**
   - **Expected answer:** **Sum** of `downtime_minutes` from `quality_metrics_daily`, grouped by plant (join `plants`).

2. **How many unit_produced events occurred in Q1 2024?**
   - **Expected answer:** A **single count** of rows in `production_events` where `event_type = 'unit_produced'` and `event_date` is in **2024-01-01** through **2024-03-31** (inclusive).


## Safety — try in Genie, then compare

1. **List critical safety incidents.**
   - **Expected answer:** Rows from `safety_incidents` where `severity = 'Critical'`, typically ordered by `incident_date` descending.

2. **How many incidents by severity?**
   - **Expected answer:** **Counts** grouped by `severity` (values include Low, Medium, High, Critical per notebook 01).


## Reference SQL — ground truth for the prompts above

Run this cell after **02_setup_data** on the **same** catalog/schema. Use the outputs as the **expected answers** when validating Genie.


In [ ]:
print("=== 1. Average OEE % by plant (2024) ===")
display(
    spark.sql(
        f"""
SELECT p.plant_name, ROUND(AVG(q.oee_score) * 100, 2) AS avg_oee_pct
FROM {fqn}.quality_metrics_daily q
JOIN {fqn}.plants p ON q.plant_id = p.plant_id
WHERE YEAR(q.date) = 2024
GROUP BY p.plant_name
ORDER BY avg_oee_pct DESC
"""
    )
)

print("=== 2. Defect rate % by production line (2024, top 15) ===")
display(
    spark.sql(
        f"""
SELECT
  pl.line_name,
  ROUND(
    SUM(CASE WHEN pe.event_type = 'defect_detected' THEN 1 ELSE 0 END) * 100.0
    / NULLIF(SUM(CASE WHEN pe.event_type = 'unit_produced' THEN 1 ELSE 0 END), 0),
    2
  ) AS defect_rate_pct
FROM {fqn}.production_events pe
JOIN {fqn}.production_lines pl ON pe.production_line_id = pl.line_id
WHERE YEAR(pe.event_date) = 2024
GROUP BY pl.line_id, pl.line_name
ORDER BY defect_rate_pct DESC
LIMIT 15
"""
    )
)

print("=== 3. First pass yield % by month (2024) ===")
display(
    spark.sql(
        f"""
SELECT
  DATE_TRUNC('month', q.date) AS month,
  ROUND(AVG(q.first_pass_yield) * 100, 2) AS avg_fpy_pct
FROM {fqn}.quality_metrics_daily q
WHERE YEAR(q.date) = 2024
GROUP BY DATE_TRUNC('month', q.date)
ORDER BY month
"""
    )
)

print("=== 4. Total downtime minutes by plant ===")
display(
    spark.sql(
        f"""
SELECT p.plant_name, ROUND(SUM(q.downtime_minutes), 2) AS total_downtime_minutes
FROM {fqn}.quality_metrics_daily q
JOIN {fqn}.plants p ON q.plant_id = p.plant_id
GROUP BY p.plant_name
ORDER BY total_downtime_minutes DESC
"""
    )
)

print("=== 5. unit_produced events in Q1 2024 ===")
display(
    spark.sql(
        f"""
SELECT CAST(COUNT(*) AS BIGINT) AS unit_produced_events_q1_2024
FROM {fqn}.production_events
WHERE event_type = 'unit_produced'
  AND event_date >= DATE('2024-01-01')
  AND event_date < DATE('2024-04-01')
"""
    )
)

print("=== 6. Critical safety incidents ===")
display(
    spark.sql(
        f"""
SELECT incident_id, incident_date, production_line_id, severity, description
FROM {fqn}.safety_incidents
WHERE severity = 'Critical'
ORDER BY incident_date DESC
"""
    )
)

print("=== 7. Incident counts by severity ===")
display(
    spark.sql(
        f"""
SELECT severity, COUNT(*) AS incident_count
FROM {fqn}.safety_incidents
GROUP BY severity
ORDER BY severity
"""
    )
)


## Quick programmatic check

Sends 3 questions to Genie via the conversation API and compares to ground truth. This confirms the space answers correctly without needing to copy-paste into the UI.


In [ ]:
import time
import re as _re

headers = {**w.config.authenticate(), "Content-Type": "application/json"}

def _extract_number(text):
    if text is None:
        return None
    nums = _re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    return float(nums[0]) if nums else None

def ask_genie_quick(question, space_id, timeout=90):
    try:
        s = requests.post(
            f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
            headers=headers, json={"content": question},
        )
        if s.status_code != 200:
            return None, f"HTTP {s.status_code}"
        d = s.json()
        cid, mid = d.get("conversation_id"), d.get("message_id")
        for _ in range(timeout // 4):
            time.sleep(4)
            p = requests.get(
                f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}",
                headers=headers,
            )
            if p.status_code != 200:
                continue
            msg = p.json()
            if msg.get("status") == "COMPLETED":
                for att in msg.get("attachments", []):
                    aid = att.get("attachment_id") or att.get("id", "")
                    if not att.get("query") or not aid:
                        continue
                    qr = requests.get(
                        f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}",
                        headers=headers,
                    )
                    if qr.status_code == 200:
                        arr = qr.json().get("statement_response", {}).get("result", {}).get("data_array", [])
                        if arr and arr[0]:
                            return _extract_number(arr[0][0]), None
                return None, "No numeric value"
            elif msg.get("status") in ("FAILED", "CANCELLED"):
                return None, msg.get("status")
        return None, "Timeout"
    except Exception as e:
        return None, str(e)[:80]

import requests

spot_checks = [
    ("How many plants are in Michigan?",
     f"SELECT COUNT(*) FROM {fqn}.plants WHERE state = 'Michigan'"),
    ("What is the total units produced in 2024?",
     f"SELECT SUM(units_produced) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024"),
    ("How many critical safety incidents are there?",
     f"SELECT COUNT(*) FROM {fqn}.safety_incidents WHERE severity = 'Critical'"),
]

print("Quick spot check: 3 questions against Configured space")
for q, gt_sql in spot_checks:
    gt = float(spark.sql(gt_sql).collect()[0][0])
    gv, err = ask_genie_quick(q, PRIMARY_SPACE_ID)
    status = "PASS" if gv is not None and abs(gv - gt) / max(abs(gt), 1) < 0.05 else "FAIL"
    print(f"  {status}: {q} (GT={gt:.0f}, Genie={gv})")
print()
print("Next: Notebook 06 -- Genie Code skills.")
